In [ ]:
from AlgorithmImports import *
import pandas as pd
import io

qb = QuantBook()

# List all trade log versions in ObjectStore
try:
    all_keys = list(qb.object_store.keys)
    log_keys = sorted([k for k in all_keys if 'trade_log' in k])
    if log_keys:
        print(f'Available trade logs: {len(log_keys)}')
        for k in log_keys:
            print(f'  {k}')
    print()
except:
    print('Could not list ObjectStore keys\n')

# Load latest trade log (change key to a timestamped version to load older runs)
key = 'trade_log.csv'
if qb.object_store.contains_key(key):
    csv_data = qb.object_store.read(key)
    df = pd.read_csv(io.StringIO(csv_data))
    print(f'Loaded {len(df)} trades from: {key}')
    display(df)

    # Summary by exit reason
    summary = df.groupby('reason').agg(
        count=('pnl', 'count'),
        avg_pnl=('pnl', 'mean'),
        total_pnl=('pnl', 'sum'),
        win_rate=('pnl', lambda x: (x > 0).mean())
    ).round(2)
    print('\n--- Exit Reason Summary ---')
    display(summary)

    # MAE/MFE summary
    print('\n--- MAE/MFE Analysis ---')
    mae_mfe = df[['direction', 'mae_pct', 'mfe_pct', 'mfe_captured_pct', 'pnl_pct']].groupby('direction').mean().round(3)
    display(mae_mfe)

    # Time of day analysis
    print('\n--- Entry Time Analysis ---')
    time_stats = df.groupby('entry_time').agg(
        count=('pnl', 'count'),
        avg_pnl=('pnl', 'mean'),
        win_rate=('pnl', lambda x: (x > 0).mean())
    ).round(2)
    display(time_stats[time_stats['count'] >= 3])
else:
    print(f'Key {key} not found in object store')

In [ ]:
# Download trade log as CSV
from IPython.display import HTML
import base64

if 'df' in dir() and df is not None and len(df) > 0:
    csv_str = df.to_csv(index=False)
    b64 = base64.b64encode(csv_str.encode()).decode()
    href = f'<a download="trade_log.csv" href="data:text/csv;base64,{b64}" target="_blank">Click here to download trade_log.csv ({len(df)} trades)</a>'
    display(HTML(href))
else:
    print('No trade data loaded. Run the cell above first.')